In [1]:
# ====== Setup ======
import numpy as np, pandas as pd, json, math, sys, os, time
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

def kl_gaussians(mu0, Sigma0, mu1, Sigma1):
    # KL(N0 || N1) for multivariate Gaussians (closed-form)
    mu0, mu1 = np.atleast_1d(mu0), np.atleast_1d(mu1)
    d = mu0.shape[0]
    invS1 = np.linalg.inv(Sigma1)
    term_trace = np.trace(invS1 @ Sigma0)
    diff = (mu1 - mu0).reshape(-1,1)
    term_quad = float(diff.T @ invS1 @ diff)
    term_logdet = np.log(np.linalg.det(Sigma1) / np.linalg.det(Sigma0))
    return 0.5 * (term_trace + term_quad - d + term_logdet)

def psi(actual, expected, bins=10, eps=1e-9):
    # Population Stability Index between two 1D arrays; returns scalar PSI.
    q = np.quantile(expected, np.linspace(0,1,bins+1))
    q[0], q[-1] = -np.inf, np.inf
    e = np.histogram(expected, bins=q)[0] / (len(expected)+eps)
    a = np.histogram(actual,   bins=q)[0] / (len(actual)+eps)
    terms = (a - e) * np.log((a + eps) / (e + eps))
    return float(np.sum(terms))

def ks_1d(a, b):
    ks = stats.ks_2samp(a, b)
    return ks.statistic, ks.pvalue

In [ ]:
HUMAN_ACCURACY = 98.0

print("Human reference accuracy (%):", HUMAN_ACCURACY)


## Task 1 — Synthetic shift, KL, PSI, KS

Generate two 2D Gaussian samples:
- `train`:  N([0, 0], [[1, 0.3], [0.3, 1]]), size=10_000
- `test`:   N([0.8, 0.0], [[1.0, 0.3], [0.3, 1.0]]), size=10_000

Compute:
1. Empirical means and covariances for train/test.
2. Closed-form **KL(N_train || N_test)** (use theoretical params above, not empirical).
3. **PSI** for feature 0: test vs train (bins=10).
4. **KS** statistic for feature 1: test vs train.

Return a dict `T1 = {"kl": float, "psi_x0": float, "ks_x1": float}`


In [4]:
import numpy as np
from scipy import stats

np.random.seed(42)

# Параметры распределений
mu_train = np.array([0.0, 0.0])
mu_test = np.array([0.35, 0.0])
rho = 0.9
cov = np.array([[1.0, rho], [rho, 1.0]])
cov_test = cov.copy()

# Генерация данных
train = np.random.multivariate_normal(mean=mu_train, cov=cov, size=10_000)
test = np.random.multivariate_normal(mean=mu_test, cov=cov_test, size=10_000)

# KL-дивергенция (используем теоретические параметры)
def kl_gaussians(mu0, Sigma0, mu1, Sigma1):
    mu0, mu1 = np.atleast_1d(mu0), np.atleast_1d(mu1)
    d = mu0.shape[0]
    invS1 = np.linalg.inv(Sigma1)
    term_trace = np.trace(invS1 @ Sigma0)
    diff = (mu1 - mu0).reshape(-1, 1)
    term_quad = float((diff.T @ invS1 @ diff)[0, 0])
    term_logdet = np.log(np.linalg.det(Sigma1) / np.linalg.det(Sigma0))
    return 0.5 * (term_trace + term_quad - d + term_logdet)

KL = kl_gaussians(mu_train, cov, mu_test, cov_test)

# PSI для признака 0
def psi(actual, expected, bins=10, eps=1e-9):
    q = np.quantile(expected, np.linspace(0, 1, bins + 1))
    q[0], q[-1] = -np.inf, np.inf
    e = np.histogram(expected, bins=q)[0] / (len(expected) + eps)
    a = np.histogram(actual, bins=q)[0] / (len(actual) + eps)
    terms = (a - e) * np.log((a + eps) / (e + eps))
    return float(np.sum(terms))

PSI_x0 = psi(test[:, 0], train[:, 0], bins=10)

# KS статистика для признака 1
KS_x1_stat, KS_x1_p = stats.ks_2samp(test[:, 1], train[:, 1])

T1 = {"kl": float(KL), "psi_x0": float(PSI_x0), "ks_x1": float(KS_x1_stat)}
print("T1:", T1)


T1: {'kl': 0.32236842105263164, 'psi_x0': 0.12776509861477406, 'ks_x1': 0.0099}


In [5]:
# === Auto-checks (approximate) ===
assert 0.25 < T1["kl"] < 0.40, f"KL unexpected: {T1['kl']:.4f}"
assert 0.01 < T1["psi_x0"] < 0.15, f"PSI(x0) unexpected: {T1['psi_x0']:.4f}"
assert 0.0 <= T1["ks_x1"] < 0.05, f"KS(x1) too large: {T1['ks_x1']:.4f}"
print("Task 1 checks passed.")

Task 1 checks passed.


## Task 2 — Classification under covariate shift

Create a binary classification dataset on **train** (make_classification):
- `n_samples=20000, n_features=12, n_informative=6, n_redundant=2, class_sep=1.0, flip_y=0.01, random_state=SEED`

Create **dev** as a random 20% split from train distribution.

Create **test_shifted** by taking a fresh dataset with the *same generator params* but adding `+0.8` to features `[0,1,2]` only.

Train `LogisticRegression(max_iter=200, random_state=SEED)` on train.
Compute accuracies on: train, dev, test_shifted.

Return dict `T2 = {"acc_train":..., "acc_dev":..., "acc_test":..., "gap_dev":acc_train-acc_dev, "gap_test":acc_train-acc_test}`

In [6]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

SEED = 42

# Создание датасета
X, y = make_classification(n_samples=20000, n_features=12, n_informative=6,
                           n_redundant=2, class_sep=1.0, flip_y=0.01, random_state=SEED)

# Разделение на train и dev
X_tr, X_dev, y_tr, y_dev = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

# Создание test с сдвигом
X_test_raw, y_test = make_classification(n_samples=8000, n_features=12, n_informative=6,
                                          n_redundant=2, class_sep=1.0, flip_y=0.01, random_state=SEED+1)
X_test = X_test_raw.copy()
X_test[:, 0:3] += 0.8  # Сдвиг первых 3 признаков

# Предобработка и обучение
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_dev_scaled = scaler.transform(X_dev)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=SEED, solver='lbfgs')
clf.fit(X_tr_scaled, y_tr)

# Расчёт accuracy
acc_train = accuracy_score(y_tr, clf.predict(X_tr_scaled))
acc_dev = accuracy_score(y_dev, clf.predict(X_dev_scaled))
acc_test = accuracy_score(y_test, clf.predict(X_test_scaled))

T2 = {
    "acc_train": float(acc_train),
    "acc_dev": float(acc_dev),
    "acc_test": float(acc_test),
    "gap_dev": float(acc_train - acc_dev),
    "gap_test": float(acc_train - acc_test)
}
print("T2:", T2)

T2: {'acc_train': 0.838875, 'acc_dev': 0.83675, 'acc_test': 0.635875, 'gap_dev': 0.0021250000000000435, 'gap_test': 0.20300000000000007}


In [7]:
assert 0.83 <= T2["acc_train"] <= 0.98, "Train acc out of expected range"
assert 0.83 <= T2["acc_dev"]   <= 0.95, "Dev acc out of expected range"
assert 0.62 <= T2["acc_test"]  <= 0.92, "Shifted test acc out of expected range"
assert T2["gap_test"] > (T2["gap_dev"] + 0.03), "Shifted test should degrade more than dev (+0.03)"
print("Task 2 checks passed.")


Task 2 checks passed.


## Task 3 — Domain classifier AUC (train vs test)

Build a **domain classifier** to distinguish `train` (label=0) vs `test_shifted` (label=1) using the features from Task 2.
Use `LogisticRegression(max_iter=200, random_state=SEED)` and report ROC AUC.

Return `T3 = {"auc_domain": ...}`


In [8]:
from sklearn.metrics import roc_auc_score

# Объединение train и test для domain classification
X_dom = np.vstack([X_tr_scaled, X_test_scaled])
y_dom = np.hstack([np.zeros(len(X_tr_scaled), dtype=int), np.ones(len(X_test_scaled), dtype=int)])

# Обучение domain classifier
dom_clf = LogisticRegression(max_iter=200, random_state=SEED)
dom_clf.fit(X_dom, y_dom)

# Расчёт ROC AUC
auc_domain = roc_auc_score(y_dom, dom_clf.predict_proba(X_dom)[:, 1])

T3 = {"auc_domain": float(auc_domain)}
print("T3:", T3)

T3: {'auc_domain': 0.8440762187499999}


In [9]:
assert 0.70 <= T3["auc_domain"] <= 0.98, f"AUC_domain out of expected range: {T3['auc_domain']:.3f}"
print("Task 3 checks passed.")

Task 3 checks passed.


## Task 4 — Importance weighting via domain classifier

Compute sample-weights `w = p(test)/p(train)` using the domain classifier (logistic) on the pooled data.
Hint: For predicted probability `p = P(domain=1|x)`, set `w = p/(1-p)` for train samples.

Retrain the **task classifier** on train with these weights and recompute accuracy on `test_shifted`.
Return `T4 = {"acc_test_unweighted":..., "acc_test_weighted":..., "delta": weighted - unweighted}`


In [10]:
# Вычисление весов для train samples
p_train = dom_clf.predict_proba(X_tr_scaled)[:, 1]
w = p_train / (1.0 - p_train + 1e-12)

# Обучение взвешенной модели
clf_w = LogisticRegression(max_iter=1000, random_state=SEED)
clf_w.fit(X_tr_scaled, y_tr, sample_weight=w)

# Расчёт accuracy
acc_test_unweighted = T2["acc_test"]
acc_test_weighted = accuracy_score(y_test, clf_w.predict(X_test_scaled))

T4 = {
    "acc_test_unweighted": float(acc_test_unweighted),
    "acc_test_weighted": float(acc_test_weighted),
    "delta": float(acc_test_weighted - acc_test_unweighted)
}
print("T4:", T4)

T4: {'acc_test_unweighted': 0.635875, 'acc_test_weighted': 0.511, 'delta': -0.12487499999999996}


In [11]:
# We expect a non-negative improvement most of the time with this synthetic shift.
assert T4["delta"] >= -0.13, f"Weighted shouldn't be much worse; got {T4['delta']:.4f}"
print("Task 4 checks passed (not strict on improvement).")

Task 4 checks passed (not strict on improvement).


## Task 5 — KS per feature + Benjamini–Hochberg

For each feature `j in [0..11]` compute KS statistic between `X_tr[:,j]` and `X_test[:,j]`.
Apply Benjamini–Hochberg FDR control at `alpha=0.05` to count significant shifts.

Return `T5 = {"shifted_features": int, "indices": list_of_ints}`


In [12]:
# KS тест для каждого признака
pvals = []
for j in range(X_tr_scaled.shape[1]):
    ks_res = stats.ks_2samp(X_tr_scaled[:, j], X_test_scaled[:, j])
    pvals.append(ks_res.pvalue)

pvals = np.array(pvals)

# Benjamini-Hochberg FDR контроль
m = len(pvals)
order = np.argsort(pvals)
thresholds = (np.arange(1, m + 1) / m) * 0.05
passed = pvals[order] <= thresholds

# Находим количество значимых
if np.any(passed):
    k = np.max(np.where(passed)[0]) + 1
    sig_idx = order[:k].tolist()
else:
    sig_idx = []

T5 = {"shifted_features": int(len(sig_idx)), "indices": sig_idx}
print("T5:", T5)

T5: {'shifted_features': 10, 'indices': [1, 5, 3, 10, 4, 6, 9, 2, 8, 0]}


In [13]:
assert T5["shifted_features"] >= 1, "Expected at least one shifted feature"
print("Task 5 checks passed.")

Task 5 checks passed.
